In [40]:
import geopandas as gpd
import plotly.express as px
import json
import pandas as pd
import numpy as np
from shapely.geometry import Point

In [41]:
lookup_2021_to_2024 = gpd.read_file("data/LSOA_(2021)_to_Electoral_Ward_(2024)_to_LAD_(2024)_Best_Fit_Lookup_in_EW.geojson")
lookup_2021_to_2023 = gpd.read_file("data/Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2023)_to_LAD_(2023)_Lookup_in_England_and_Wales.geojson")
lookup_2021_to_2022 = gpd.read_file("data/Lower_Layer_Super_Output_Area_(2021)_to_Ward_(2022)_to_LAD_(2022)_Lookup_in_England_and_Wales_v3.geojson")

lookup_2011_to_2021 = pd.read_excel("data/LSOA11_WD21_LAD21_EW_LU_V2.xlsx")
# lookup_2011_to_2021 = gpd.read_file("data/LSOA11_WD21_LAD21_EW_LU_V2.xlsx")
lookup_2011_to_2020 = gpd.read_file("data/LSOA11_WD20_LAD20_EW_LU_v2_f514a75a131249caa65227cdc6275a21_4644476982633191757.geojson")
lookup_2011_to_2019 = gpd.read_file("data/LSOA11_WD19_LAD19_EW_LU_cbf3896924a74e58ac96b7ec66a34071_6359649327362015065.geojson")

In [42]:
lookup_2021_to_2024_london = lookup_2021_to_2024[lookup_2021_to_2024['LAD24CD'].str.startswith('E09')]
lookup_2021_to_2023_london = lookup_2021_to_2023[lookup_2021_to_2023['LAD23CD'].str.startswith('E09')]
lookup_2021_to_2022_london = lookup_2021_to_2022[lookup_2021_to_2022['LAD22CD'].str.startswith('E09')]

lookup_2011_to_2021_london = lookup_2011_to_2021[lookup_2011_to_2021['LAD21CD'].str.startswith('E09')]
lookup_2011_to_2020_london = lookup_2011_to_2020[lookup_2011_to_2020['LAD20CD'].str.startswith('E09')]
lookup_2011_to_2019_london = lookup_2011_to_2019[lookup_2011_to_2019['LAD19CD'].str.startswith('E09')]

In [43]:
wards_dec2024 = gpd.read_file("data/Wards_December_2024_Boundaries_UK_BFC_7247148252775165514.geojson")
wards_may2024 = gpd.read_file("data/Wards_May_2024_Boundaries_UK_BFE_2105195262474198835.geojson")
wards_dec2023 = gpd.read_file("data/Wards_December_2023_Boundaries_UK_BFC_-3859483768213475321.geojson")
wards_may2023 = gpd.read_file("data/WD_MAY_2023_UK_BFC_2710077981141295185.geojson")
wards_dec2022 = gpd.read_file("data/Wards_December_2022_Boundaries_GB_BFC_-307155175152131259.geojson")
wards_dec2021 = gpd.read_file("data/Wards_December_2021_GB_BFC_2022_-8743003948094954925.geojson")
wards_dec2020 = gpd.read_file("data/Wards_December_2020_UK_BFC_2022_5104647273706858769.geojson")
wards_dec2019 = gpd.read_file("data/Wards_December_2019_Boundaries_UK_BFC_2022_1904482841917252715.geojson")

In [44]:
lsoas_dec2021 = gpd.read_file("data/Lower_layer_Super_Output_Areas_(December_2021)_Boundaries_EW_BFC_(V10).geojson")
lsoas_dec2011 = gpd.read_file("data/LSOAs.geojson")

In [45]:
def filter_london_wards(wards_gdf, london_boundary_path):
    """
    Filters a ward GeoDataFrame by checking if LONG/LAT fall inside Greater London boundary.

    Parameters:
        wards_gdf (GeoDataFrame): Input wards GeoDataFrame (must contain LONG and LAT columns).
        london_boundary_path (str): Path to Greater London boundary shapefile.

    Returns:
        GeoDataFrame: Filtered wards with original geometry intact.
    """
    # Create a temporary Point geometry from LONG/LAT, don't overwrite existing geometry
    wards_gdf = wards_gdf.copy()  # avoid modifying original df
    wards_gdf['point_geom'] = wards_gdf.apply(lambda row: Point(row['LONG'], row['LAT']), axis=1)
    
    # Create a GeoSeries for points with correct CRS
    points = gpd.GeoSeries(wards_gdf['point_geom'], crs='EPSG:4326')

    # Load London boundary and convert CRS to match points
    london_boundary = gpd.read_file(london_boundary_path)
    london_boundary = london_boundary.to_crs(points.crs)

    # Get combined London polygon using union_all
    london_polygon = london_boundary.geometry.union_all()

    # Filter rows where points fall within London polygon
    mask = points.within(london_polygon)

    # Return filtered GeoDataFrame, keeping original geometry column
    return wards_gdf.loc[mask].drop(columns=['point_geom'])

In [46]:
london_boundary_path = "data/London_GLA_boundary.shp"

wards_dec2024_london = filter_london_wards(wards_dec2024, london_boundary_path)
wards_may2024_london = filter_london_wards(wards_may2024, london_boundary_path)
wards_dec2023_london = filter_london_wards(wards_dec2023, london_boundary_path)
wards_may2023_london = filter_london_wards(wards_may2023, london_boundary_path)
wards_dec2022_london = filter_london_wards(wards_dec2022, london_boundary_path)
wards_dec2021_london = filter_london_wards(wards_dec2021, london_boundary_path)
wards_dec2020_london = filter_london_wards(wards_dec2020, london_boundary_path)
wards_dec2019_london = filter_london_wards(wards_dec2019, london_boundary_path)

In [47]:
wards_dec2024_london

,FID,WD24CD,WD24NM,WD24NMW,LAD24CD,LAD24NM,LAD24NMW,BNG_E,BNG_N,LONG,LAT,GlobalID,geometry
1104,1105,E05009288,Aldersgate,,E09000001,City of London,,532171,181720,-0.09642,51.51895,313ea69e-d2cb-41e1-a0fe-6e9214327662,"POLYGON ((-0.09536 51.51749, -0.09543 51.51716..."
1105,1106,E05009289,Aldgate,,E09000001,City of London,,533397,181175,-0.07897,51.51376,190ba6c2-4a6d-4649-9cab-0cced4418d49,"POLYGON ((-0.07789 51.51582, -0.07782 51.51573..."
1106,1107,E05009290,Bassishaw,,E09000001,City of London,,532438,181495,-0.09266,51.51686,8a78c937-97aa-4042-85fa-28253dd95db5,"POLYGON ((-0.09119 51.51808, -0.09107 51.51805..."
1107,1108,E05009291,Billingsgate,,E09000001,City of London,,533151,180754,-0.08267,51.51004,2fc3ae18-0811-49d3-8760-9a9f4a55c260,"POLYGON ((-0.0803 51.50807, -0.08033 51.50808,..."
1108,1109,E05009292,Bishopsgate,,E09000001,City of London,,533207,181664,-0.08152,51.51820,f5d63b87-4b75-4812-bd73-6ab02376078d,"POLYGON ((-0.07853 51.52151, -0.07853 51.52151..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5011,5012,E05014115,Streatham St Leonard's,,E09000022,Lambeth,,529944,171661,-0.13220,51.42906,f1599e75-82d5-484f-bf0b-e617d476ddc8,"POLYGON ((-0.1269 51.43818, -0.12688 51.43818,..."
5012,5013,E05014116,Streatham Wells,,E09000022,Lambeth,,530653,171913,-0.12191,51.43117,df3be6ae-322f-4edb-91b8-22e19507f36c,"POLYGON ((-0.11781 51.43305, -0.11769 51.43303..."
5013,5014,E05014117,Vauxhall,,E09000022,Lambeth,,530507,178186,-0.12170,51.48757,176ca2df-040e-4793-8e44-2718db69ef16,"POLYGON ((-0.11819 51.4924, -0.11827 51.4923, ..."
5014,5015,E05014118,Waterloo & South Bank,,E09000022,Lambeth,,531026,179803,-0.11363,51.50199,32dfc21b-6626-4d30-a8fe-2c95a1ebff09,"POLYGON ((-0.10891 51.50842, -0.10889 51.50832..."


In [48]:
# Create a temporary 'point_geom' column from LONG/LAT without overwriting original geometry
lsoas_dec2021 = lsoas_dec2021.copy()
lsoas_dec2021['point_geom'] = lsoas_dec2021.apply(lambda row: Point(row['LONG'], row['LAT']), axis=1)

# Create GeoSeries for points with CRS
points = gpd.GeoSeries(lsoas_dec2021['point_geom'], crs='EPSG:4326')

# Load London boundary
london_boundary = gpd.read_file(london_boundary_path)
london_boundary = london_boundary.to_crs(points.crs)

# Get combined polygon of London boundary using union_all()
london_polygon = london_boundary.geometry.union_all()

# Filter rows where point_geom is within London polygon
mask = points.within(london_polygon)

# Filter original GeoDataFrame and drop temporary column
london_lsoas_dec2021 = lsoas_dec2021.loc[mask].drop(columns=['point_geom'])
london_lsoas_dec2011 = lsoas_dec2011

In [ ]:
london_wards = wards_dec2024_london.to_crs(epsg=4326)

fig = px.choropleth_map(
    london_wards,
    geojson=json.loads(london_wards.to_json()),
    locations='WD24CD',
    featureidkey="properties.WD24CD",
    color_discrete_sequence=["lightblue"],
    map_style="open-street-map",
    zoom=9,
    center={"lat": 51.5072, "lon": -0.1276},
    opacity=0.6,
    height=600,
    hover_name='WD24NM'
)

fig.update_layout(title='London Wards')
fig.show()

In [ ]:
london_lsoas = london_lsoas_dec2021.to_crs(epsg=4326)

fig = px.choropleth_map(
    london_lsoas,
    geojson=json.loads(london_lsoas.to_json()),
    locations='LSOA21CD',
    featureidkey="properties.LSOA21CD",
    color_discrete_sequence=["lightblue"],
    map_style="open-street-map",
    zoom=9,
    center={"lat": 51.5072, "lon": -0.1276},
    opacity=0.6,
    height=600,
    hover_name='LSOA21NM'
)

fig.update_layout(title='London LSOAs 2021')
fig.show()


In [51]:
def compute_overlap(lsoa, lsoa_cd, lsoa_nm, ward, ward_cd, ward_nm):
    london_lsoas = lsoa.to_crs(epsg=27700)
    london_wards = ward.to_crs(epsg=27700)

    joined = gpd.sjoin(london_lsoas, london_wards, how='inner', predicate='intersects')

    joined["intersection"] = joined.apply(
        lambda row: row["geometry"].intersection(london_wards.loc[row["index_right"], "geometry"]),
        axis=1
    )

    joined["intersection_area"] = joined["intersection"].area

    joined["lsoa_area"] = joined["geometry"].area

    joined["lsoa_area_pct_in_ward"] = (joined["intersection_area"] / joined["lsoa_area"]) * 100

    result = joined[[
        lsoa_cd, lsoa_nm, ward_cd, ward_nm, "lsoa_area_pct_in_ward"
    ]].sort_values(by=[lsoa_cd, "lsoa_area_pct_in_ward"], ascending=[True, False])
    
    # keep highest
    ward_counts = result.groupby(lsoa_cd)[ward_nm].nunique()

    df_sorted = result.sort_values([lsoa_cd, 'lsoa_area_pct_in_ward'], ascending=[True, False])

    df_max = df_sorted.drop_duplicates(subset=lsoa_cd, keep='first').copy()

    df_max['ward_count'] = df_max[lsoa_cd].map(ward_counts)

    df_max = df_max.sort_values(lsoa_cd)

    df_max_sorted = df_max.sort_values(by='lsoa_area_pct_in_ward', ascending=False)

    return df_max_sorted

In [52]:
wards_old = gpd.read_file("data/wards.geojson")
wards2019 = gpd.read_file("data/wards2019.geojson")

In [53]:
lsoa_list = [london_lsoas_dec2021, london_lsoas_dec2011]
ward_list = [wards_dec2024_london, wards_may2024_london, wards_dec2023_london, wards_may2023_london, wards_dec2022_london, wards_dec2021_london, wards_dec2020_london, wards_dec2019_london, wards_old, wards2019]

for lsoa in lsoa_list:
    for ward in ward_list:
        overlap_df = compute_overlap(lsoa, lsoa.columns[1], lsoa.columns[2], ward, ward.columns[1], ward.columns[2])
        print(lsoa.columns[1], ward.columns[1], len(overlap_df[overlap_df['lsoa_area_pct_in_ward'] < 90]))

LSOA21CD WD24CD 1069
LSOA21CD WD24CD 1069
LSOA21CD WD23CD 1069
LSOA21CD WD23CD 1069
LSOA21CD WD22NM 1068
LSOA21CD WD21CD 267
LSOA21CD WD20CD 267
LSOA21CD WD19CD 268
LSOA21CD GSS_Code 4
LSOA21CD GSS_CODE 268
LSOA11NM WD24CD 1063
LSOA11NM WD24CD 1063
LSOA11NM WD23CD 1063
LSOA11NM WD23CD 1063
LSOA11NM WD22NM 1062
LSOA11NM WD21CD 262
LSOA11NM WD20CD 262
LSOA11NM WD19CD 261
LSOA11NM GSS_Code 4
LSOA11NM GSS_CODE 261


In [54]:
# # Plot LSOA and its intersecting wards


# lsoa_code = "E01003986"

# london_lsoas = lsoa.to_crs(epsg=4326)
# london_wards = ward.to_crs(epsg=4326)

# lsoa_target = london_lsoas[london_lsoas.columns[1] == lsoa_code].copy()

# intersecting_wards = gpd.sjoin(london_wards, lsoa_target, how='inner', predicate='intersects').copy()

# lsoa_target["type"] = "LSOA"
# intersecting_wards["type"] = "Ward"

# lsoa_target["Name"] = lsoa_target["LSOA21NM"]
# intersecting_wards["Name"] = intersecting_wards["WD24NM"]

# combined = pd.concat([
#     lsoa_target[["geometry", "Name", "type"]],
#     intersecting_wards[["geometry", "Name", "type"]]
# ])


# geojson_combined = json.loads(combined.to_json())

# fig = px.choropleth_map(
#     combined,
#     geojson=geojson_combined,
#     locations=combined.index,
#     color="type",
#     hover_name="Name",
#     map_style="open-street-map",
#     zoom=9,
#     center={"lat": 51.5072, "lon": -0.1276},
#     opacity=0.5,
#     height=600
# )

# fig.update_layout(title=f'LSOA {lsoa} and Intersecting Wards')
# fig.show()